# Orchestrateur du Projet ELK - Movies Data Platform

Ce notebook rassemble **toutes les tâches** nécessaires au lancement, à la configuration et à l'arrêt du projet ELK. 
Il répond aux exigences de déploiement reproductible (F1) et gère l'automatisation du mapping strict (F4).

## 1. Démarrage de la Stack ELK
Lancement des conteneurs via Docker Compose en arrière-plan.

In [ ]:
# !docker compose up -d (A lancer depuis un terminal, pas dans Jupyter)

## 2. Configuration d'Elasticsearch (F4 - Mapping & Analyzer)
On attend qu'Elasticsearch démarre, puis on déploie automatiquement l'**Index Template**. Ainsi, dès que Logstash va ingérer le CSV, les données adopteront notre schéma.

In [ ]:
from elasticsearch import Elasticsearch
import json
es = Elasticsearch('http://elasticsearch:9200')

In [ ]:
# Définition de l'index template qui s'appliquera automatiquement aux nouveaux index
index_template = {
  # Ce template cible l'index movies_clean
  "index_patterns": ["movies_clean*"],
  "template": {
    "settings": {
      "analysis": {
        "analyzer": {
          # Configuration de l'analyseur
          "movie_title_analyzer": {
            "type": "custom",
            "tokenizer": "standard",
            # texte en minuscule, on retire les accents et on filtre les mots de liaisons.
            "filter": ["lowercase", "asciifolding", "english_stop"]
          }
        },
        "filter": {
          # dupprimer les mots de liaison anglais
          "english_stop": {
            "type": "stop",
            "stopwords": "_english_"
          }
        }
      }
    },
    # Définition du mapping
    "mappings": {
      "properties": {
        "id": { "type": "keyword" },
        "title": { 
          "type": "text", 
          "analyzer": "movie_title_analyzer",
          "fields": {
            "keyword": {
              "type": "keyword",
              "ignore_above": 256
            }
          }
        },
        # Champs de catégories et métadonnées techniques traités comme des mots-clés
        "genres": { "type": "keyword" },
        "original_language": { "type": "keyword" },
        "overview": { 
          "type": "text",
          # Utilisation de l'analyseur standard anglais
          "analyzer": "english"
        },
        # Données numériques pour les scores de popularité et les statistiques
        "popularity": { "type": "float" },
        "production_companies": { "type": "keyword" },
        "release_date": { "type": "date" },
        # Valeurs financières et durée traitées en nombres flottants
        "budget": { "type": "float" },
        "revenue": { "type": "float" },
        "runtime": { "type": "float" },
        "status": { "type": "keyword" },
        "tagline": { "type": "text" },
        "vote_average": { "type": "float" },
        "vote_count": { "type": "integer" },
        # Chemins d'accès et listes de recommandations comme tel
        "credits": { "type": "keyword" },
        "keywords": { "type": "keyword" },
        "poster_path": { "type": "keyword" },
        "backdrop_path": { "type": "keyword" },
        "recommendations": { "type": "keyword" }
      }
    }
  }
}

try:
    # Envoi du template
    es.indices.put_index_template(
        name="movies_template",
        index_patterns=index_template.get("index_patterns"),
        template=index_template.get("template", {})
    )
    print("[OK] Template pour movies_clean déployé.")
except Exception as e:
    print("[ERREUR] lors du déploiement du template pour movies_clean.", e)

## 3. Check d'ingestion
Vérification de la qualité des données ingérées

In [ ]:
print("=== VERIFICATIONS D'INGESTION ===\n")

# Mapping
mapping = es.indices.get_mapping(index="movies_clean")
print(f"[OK] Index 'movies_clean' accessible et mappé. {json.dumps(mapping.body, indent=4, ensure_ascii=False)}")

# Nombres totaux
raw_count = es.count(index="movies_raw")['count']
clean_count = es.count(index="movies_clean")['count']
print(f"[INFO] Count brut: {raw_count}")
print(f"[INFO] Count nettoyé: {clean_count} (soit {raw_count - clean_count} anomalies supprimées)\n")

# Echantillons
raw_sample = es.search(index="movies_raw", size=1)
clean_sample = es.search(index="movies_clean", size=1)
print("[INFO] Exemple brut (popularity) :", raw_sample['hits']['hits'][0]['_source'].get('popularity'), "(Type reel :", type(raw_sample['hits']['hits'][0]['_source'].get('popularity')).__name__, ")")
print("[INFO] Exemple propre (popularity) :", clean_sample['hits']['hits'][0]['_source'].get('popularity'), "(Type reel :", type(clean_sample['hits']['hits'][0]['_source'].get('popularity')).__name__, ")\n")

# Vérification si le filtrage est fonctionnel (Devrait être 0)
q_anomalies = {
    "query": {
        "bool": {
            "should": [
                { "bool": { "must_not": { "exists": { "field": "title.keyword" } } } },
                { "bool": { "must_not": { "exists": { "field": "release_date" } } } }
            ],
            "minimum_should_match": 1
        }
    }
}
print("[INFO] Lignes avec titre ou date manquante (devrait être 0) :", es.count(index="movies_clean", body=q_anomalies)['count'])

# Vérification des opérations numériques (gte 2000)
q_range = {
    "query": {
        "range": {
            "popularity": { "gte": 2000 }
        }
    },
    "_source": ["title", "popularity"]
}
res_range = es.search(index="movies_clean", body=q_range)
print(f"[INFO] Films tres populaires (>=2000) Si ne fonctionne pas, le typage/mapping n'a pas fonctionnée : {res_range['hits']['total']['value']}")

# Vérification de la casse (Devrait être 0)
q_regex = {
    "query": {
        "regexp": {
            "original_language.keyword": ".*[A-Z].*"
        }
    }
}
print("[INFO] Films avec langue contenant des majuscules (doit être 0) :", es.count(index="movies_clean", body=q_regex)['count'])

## 4. Réinitialisation & Ingestion
Vider les données existantes et forcer Logstash à relire le CSV depuis le début avec le nouveau modèle.

In [ ]:
print("[INFO] Suppression de l'index movies_clean...")
try:
    es.indices.delete(index="movies_clean", ignore_unavailable=True)
    es.indices.delete(index="movies_raw", ignore_unavailable=True)
    print("[OK] Index supprimés.")
except Exception as e:
    print("[ERREUR] lors de la suppression :", e)

print("[INFO] Vous devez redémarrer le conteneur Logstash manuellement")
print("Commande à taper : docker restart logstash")